# Football Predictor -- Notebook version

This runs the same pooled Dixon-Coles model as `football_predictor.py`, but
as plain function/class calls instead of through the command-line interface
-- no `argparse`, so it works fine inside Jupyter.

**Requirement:** `football_predictor.py` must be in the same folder as this
notebook (or update the `sys.path.insert` line below to point at it).

Cells are meant to be run top to bottom. The data-download cell can take a
few minutes the first time (10 seasons x 4 divisions = 40 file downloads).

In [ ]:
import sys
sys.path.insert(0, ".")  # change this if football_predictor.py lives elsewhere

from football_predictor import (
    load_multi_division_historical, load_division_seasons, load_fixtures_from_site,
    DixonColes, simulate_season, simulate_cup, build_current_table,
    print_lsc_table, print_match_prediction, DIVISION_NAMES,
    predict_fixtures, resolve_team_name,
    position_probability_table, style_position_table,
)
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

## Step 1 -- Download and pool historical results

Pulls the last `N_SEASONS` seasons for each division in `DIVISIONS` from
football-data.co.uk and concatenates them into one dataframe.

In [ ]:
DIVISIONS = ["E0", "E1", "E2", "E3"]   # Prem, Championship, League One, League Two
N_SEASONS = 10

data = load_multi_division_historical(DIVISIONS, N_SEASONS)
print(f"Loaded {len(data)} matches across {data['Division'].nunique()} divisions")
data.head()

## Step 2 -- Fit the pooled Dixon-Coles model

One attack/defense rating per team, shared across whichever division(s) it
played in -- promoted/relegated teams link the divisions' scales together.

In [ ]:
model = DixonColes().fit(data)

print("Converged:", model._fit_success, " rho:", round(model.params["rho"], 3))
for dv in model.divisions:
    print(f"  {DIVISION_NAMES.get(dv, dv):<16} home_adv={model.params['home_adv'][dv]:+.3f}  "
          f"tempo={model.params['tempo'][dv]:+.3f}")

## Step 3 -- League Strength Coefficients

How strong is a typical team in each division, on the pooled common scale.

In [ ]:
lsc = model.league_strength_coefficients()
lsc

## Step 3b -- List all available teams

Every team the model can be used with (i.e. valid inputs to `predict_match`,
`simulate_season`, `simulate_cup`). Team names must match football-data.co.uk's
exact spelling -- use this list to look them up rather than guessing.

`DivisionsPlayed` shows every division a team appeared in during the training
window; teams with more than one are the promoted/relegated "bridge" teams
that link the divisions' rating scales together (see Step 3).

In [ ]:
teams_df = model.list_teams()
print(f"{len(teams_df)} teams available")
teams_df

Filter to a single division, or search by name:

In [ ]:
model.list_teams(division="E0")   # just the Premier League

# teams_df[teams_df["Team"].str.contains("United", case=False)]   # example name search

## Step 3c -- Fix teams labeled in the wrong division

`CurrentDivision` in `list_teams()` is set to whichever division a team's
**most recent recorded match** was in. Two things commonly make this look
wrong:

1. **Timing** -- a team was promoted/relegated for the upcoming season, but
   that season's fixtures haven't been played yet (or football-data.co.uk
   hasn't posted the new season's file), so the model still shows last
   season's division. Fix with `override_team_division`.
2. **Spelling drift** -- the same real club is spelled slightly differently
   in different divisions' source files (e.g. `"Nott'm Forest"` vs
   `"Nottingham Forest"`), so the model silently treats them as two
   different teams. Check with `find_similar_team_names`.

In [ ]:
# 1) Manually correct any teams you know are mislabeled, e.g. confirmed
#    close-season promotions/relegations that haven't been played yet:
model.override_team_division({
    "Leicester": "E0",     # example: promoted back to the Prem
    # "Southampton": "E1",  # example: relegated
})

model.list_teams(division="E0")   # confirm the fix took

In [ ]:
# 2) Check for teams that might be the same club under two different
#    spellings -- if you spot a real duplicate here, standardise the
#    spelling in your source CSV(s) and refit, rather than overriding.
model.find_similar_team_names()

## Step 4 -- Predict a single match

Works for a normal same-division match...

In [ ]:
pred = model.predict_match("Arsenal", "Chelsea")
print_match_prediction(pred)

pd.DataFrame(pred["top_scorelines"])

...and equally for a cross-division match (e.g. a cup tie), since ratings sit
on one shared scale.

In [ ]:
cup_pred = model.predict_match("Arsenal", "Wrexham")
print_match_prediction(cup_pred)

## Step 4b -- Predict a whole file of fixtures (e.g. this week's games)

Instead of calling `predict_match` one game at a time, build (or load) a
small table of `HomeTeam,AwayTeam` fixtures and run `predict_fixtures` once.
It prints the full breakdown for every match (same as `print_match_prediction`)
and returns a tidy summary table you can sort, filter, or export.

**Option A -- type the fixtures in directly:**

In [ ]:
this_weeks_games = pd.DataFrame({
    "HomeTeam": ["Hull", "Arsenal", "Leeds"],
    "AwayTeam": ["Man United", "Chelsea", "Sunderland"],
})

summary = predict_fixtures(model, this_weeks_games)
summary

**Option B -- load the fixtures from a CSV file** (two columns: `HomeTeam`,
`AwayTeam`). This is the easiest way to do a whole matchday/gameweek at once
-- just build the CSV in Excel/Sheets and point at it here:

In [ ]:
this_weeks_games = pd.read_csv("this_weeks_games.csv")   # your own file, HomeTeam/AwayTeam columns
summary = predict_fixtures(model, this_weeks_games)
summary

If a team name doesn't match football-data.co.uk's exact spelling,
`predict_fixtures` won't crash the whole batch -- it skips that row, prints a
warning with spelling suggestions (via `resolve_team_name`), and marks it in
an `Error` column in the returned table so you can fix and re-run just that
row:

In [ ]:
resolve_team_name(model, "Man Utd")   # example: raises with suggestions if the spelling is off

## Step 5 -- Simulate the rest of a league season (10,000 runs)

`played` = results already in the bag this season. `fixtures` = what's left
to simulate.

**Note:** football-data.co.uk's `fixtures.csv` only covers the next couple of
weeks, not a whole remaining season. For a true full-season simulation,
build your own `fixtures` dataframe (two columns: `HomeTeam`, `AwayTeam`) --
e.g. `pd.read_csv("my_fixtures.csv")` -- using team names spelled exactly as
football-data.co.uk spells them.

In [ ]:
DIVISION_TO_SIMULATE = "E0"
CURRENT_SEASON_START = 2025  # 2025 -> 2025/26 season; adjust as needed

current_season_data = load_division_seasons(DIVISION_TO_SIMULATE, 1, current_start_year=CURRENT_SEASON_START)
played = current_season_data[current_season_data["SeasonStartYear"] == CURRENT_SEASON_START]
print(f"Matches already played this season: {len(played)}")

# Option A: near-term fixtures from football-data.co.uk (next couple of weeks only)
fixtures = load_fixtures_from_site(DIVISION_TO_SIMULATE)

# Option B (recommended for a full-season sim): load your own fixture list instead
# fixtures = pd.read_csv("my_fixtures.csv")

print(f"Fixtures to simulate: {len(fixtures)}")
fixtures.head()

In [ ]:
table = simulate_season(model, played, fixtures, division=DIVISION_TO_SIMULATE, n_sims=10000)
table.round(1)

## Step 5b -- Full finishing-position probability grid

`simulate_season` already tracks, across all 10,000 runs, exactly which
position every team finished in -- the summary table above just collapses
that down to a few headline numbers (title/top-4/relegation probability).
This pulls out the complete grid: probability of finishing in *every* exact
position, 1st through last -- similar to the season-projection tables you'll
see on football analytics sites, but showing every column instead of just
cumulative top-N bands.

In [ ]:
pos_grid = position_probability_table(table)
pos_grid

And the same grid rendered as a colour-graded heatmap (blue = European/
qualification zone, red = relegation zone) -- adjust `european_spots` /
`relegation_spots` to match the division you're simulating:

In [ ]:
style_position_table(table, european_spots=5, relegation_spots=3)

## Step 6 -- Simulate a knockout cup bracket (10,000 runs)

Round-1 ties as a list of `(Home, Away)` tuples -- must be a power of 2 (8,
16, 32, 64...). Winner of `ties[0]` plays winner of `ties[1]` in round 2, and
so on (standard bracket seeding). Works across divisions since all ratings
are on the same pooled scale.

In [ ]:
# Replace with a real round of fixtures, or load from CSV:
# ties_df = pd.read_csv("cup_fixtures.csv")
# round1_ties = list(ties_df[["HomeTeam", "AwayTeam"]].itertuples(index=False, name=None))

round1_ties = [
    ("Arsenal", "Wrexham"),
    ("Man City", "Salford City"),
    ("Liverpool", "Port Vale"),
    ("Chelsea", "Colchester United"),
    ("Newcastle", "Barnsley"),
    ("Tottenham", "Bradford City"),
    ("Man United", "Exeter City"),
    ("Aston Villa", "Notts County"),
]

cup_result = simulate_cup(model, round1_ties, n_sims=10000)
cup_result.round(1)

## Notes / limitations

- **Team names** must match football-data.co.uk's exact spelling (e.g. "Man
  United", "Man City", "Nott'm Forest"). If `predict_match` raises `Unknown
  team(s)`, check `model.teams` for the exact spelling used.
- **Full fixture lists** and **cup draws** aren't published in advance by
  football-data.co.uk -- supply your own CSVs for a true full-season or
  full-bracket simulation (see Steps 5 and 6 above).
- **Two-legged cup ties** aren't modeled -- each tie is a single match, with
  draws resolved via a lightly skill-weighted simulated penalty shootout.
- Re-run Steps 1-2 only when you want to refresh the training data /
  refit the model; everything after that (predictions, season sim, cup sim)
  reuses the already-fitted `model` object and re-runs quickly.